# Task 06: Async RAG Pipeline (Google Colab)


In [1]:
!pip -q install sentence-transformers faiss-cpu transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 37.9 MB/s eta 0:00:00


In [2]:
import asyncio
import numpy as np
import faiss

from dataclasses import dataclass
from sentence_transformers import SentenceTransformer, CrossEncoder


In [3]:
@dataclass
class Chunk:
    id:int
    text:str


In [4]:
class AsyncRAGPipeline:

    def __init__(self,chunks):
        self.chunks=chunks

    def setup_models(self):
        self.embedder=SentenceTransformer("all-MiniLM-L6-v2")
        self.reranker=CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

    def build_index(self):
        texts=[c.text for c in self.chunks]
        embeddings=self.embedder.encode(texts,convert_to_numpy=True)
        self.embeddings=embeddings.astype("float32")
        self.index=faiss.IndexFlatL2(self.embeddings.shape[1])
        self.index.add(self.embeddings)

    async def retrieve(self,query,k=3):
        q=self.embedder.encode([query],convert_to_numpy=True).astype("float32")
        distances,indices=self.index.search(q,k)
        return [self.chunks[i] for i in indices[0]]

    async def rerank(self,query,retrieved):
        pairs=[[query,c.text] for c in retrieved]
        scores=self.reranker.predict(pairs)
        ranked=[x for _,x in sorted(zip(scores,retrieved),reverse=True)]
        return ranked

    async def synthesize_answer(self,query,ranked):
        context="\n".join([c.text for c in ranked])
        return f"Question: {query}\n\nRelevant Context:\n{context}"


In [5]:
chunks=[
Chunk(1,"RAG combines retrieval with language models."),
Chunk(2,"FAISS is used for fast similarity search."),
Chunk(3,"Sentence Transformers generate embeddings."),
Chunk(4,"CrossEncoder reranks retrieved documents."),
Chunk(5,"Asyncio allows asynchronous execution in Python.")
]

In [6]:
pipeline=AsyncRAGPipeline(chunks)
pipeline.setup_models()
pipeline.build_index()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [7]:
async def main():
    query="What is RAG and how does FAISS help?"

    retrieved=await pipeline.retrieve(query)

    print("Retrieved Chunks")
    for c in retrieved:
        print("-",c.text)

    ranked=await pipeline.rerank(query,retrieved)

    print("\nReranked Chunks")
    for c in ranked:
        print("-",c.text)

    answer=await pipeline.synthesize_answer(query,ranked)

    print("\nFinal Answer")
    print(answer)

await main()


Retrieved Chunks
- RAG combines retrieval with language models.
- FAISS is used for fast similarity search.
- Asyncio allows asynchronous execution in Python.

Reranked Chunks
- FAISS is used for fast similarity search.
- RAG combines retrieval with language models.
- Asyncio allows asynchronous execution in Python.

Final Answer
Question: What is RAG and how does FAISS help?

Relevant Context:
FAISS is used for fast similarity search.
RAG combines retrieval with language models.
Asyncio allows asynchronous execution in Python.
